# 25. Four-Brand Portfolio Comparison

## Objective

This notebook compares the four completed brand case studies used in Chapter 4: Hermès, Maison Francis Kurkdjian, Nasomatto and Nicolai Parfumeur Créateur. It produces portfolio composition, accord diversity and concentration, cross-brand overlap, and gender-specific diversity outputs.

## Data source and interpretation

The primary source is `main.fragrance_analytics.fragrantica_perfumes`. Records are retained as represented in the source table; no product-level deduplication or reconciliation is attempted. Counts therefore describe observed Fragrantica source records, not necessarily unique compositions or complete manufacturer catalogues.

Fragrantica ratings represent individual evaluations within the Fragrantica community. Rating counts represent engagement on that platform. Neither measure is interpreted as sales, market share, consumer demand or purchase intent.

Accord order is preserved. The first accord is the dominant accord, the first two form the ordered dyad, and the first three form the ordered triad. Missing accord positions are not imputed.

The notebook does not modify the source table or existing case-study notebooks.

In [ ]:
# 1. Load the Fragrantica source table.

from pyspark.sql import functions as F

df_fragrantica = spark.table("main.fragrance_analytics.fragrantica_perfumes")

print("Fragrantica source records:", df_fragrantica.count())

In [ ]:
# 2. Isolate the four case-study portfolios.

brand_aliases = {
    "Hermès": ["hermes", "hermès"],
    "Maison Francis Kurkdjian": ["maison francis kurkdjian"],
    "Nasomatto": ["nasomatto"],
    "Nicolai Parfumeur Créateur": [
        "nicolai parfumeur createur",
        "nicolaï parfumeur créateur",
        "nicolai parfumeur créateur"
    ]
}

brand_rows = [
    (brand, alias)
    for brand, aliases in brand_aliases.items()
    for alias in aliases
]

brand_alias_df = spark.createDataFrame(
    brand_rows,
    ["comparison_brand", "brand_key"]
)

source_records = (
    df_fragrantica
    .withColumn("brand_key", F.lower(F.trim(F.col("brand_name"))))
    .join(brand_alias_df, "brand_key", "inner")
    .select(
        "id",
        "fragrance_name_clean",
        "brand_name",
        "comparison_brand",
        "gender",
        "rating_value",
        "rating_count",
        "main_accords"
    )
    .dropDuplicates(["id"])
)

print("Selected source records:", source_records.count())

display(
    source_records
    .groupBy("comparison_brand")
    .agg(F.countDistinct("id").alias("fragrance_count"))
    .orderBy("comparison_brand")
)

In [ ]:
# 3. Parse and normalize ordered accord profiles.

accords_clean = F.trim(F.col("main_accords"))

valid_accords = (
    F.col("main_accords").isNotNull()
    & (accords_clean != "")
    & (~F.lower(accords_clean).isin("null", "none", "nan", "n/a", "[]"))
)

accord_array = F.from_json(
    F.regexp_replace(
        F.regexp_replace(F.col("main_accords"), "'", '"'),
        r"\bNone\b",
        "null"
    ),
    "array<string>"
)

portfolio_profiles = (
    source_records
    .withColumn("accords", F.when(valid_accords, accord_array))
    .withColumn(
        "accords",
        F.transform("accords", lambda x: F.lower(F.trim(x)))
    )
    .withColumn("dominant_accord", F.get("accords", 0))
    .withColumn("second_accord", F.get("accords", 1))
    .withColumn("third_accord", F.get("accords", 2))
    .withColumn("gender_key", F.lower(F.trim(F.col("gender"))))
)

display(
    portfolio_profiles
    .select(
        "comparison_brand",
        "fragrance_name_clean",
        "gender",
        "dominant_accord",
        "second_accord",
        "third_accord"
    )
    .orderBy("comparison_brand", "gender", "fragrance_name_clean")
)

print(
    "Records with a dominant accord:",
    portfolio_profiles.filter(F.col("dominant_accord").isNotNull()).count()
)

In [ ]:
# 4. Portfolio overview and accord diversity.

portfolio_overview = (
    portfolio_profiles
    .groupBy("comparison_brand")
    .agg(
        F.countDistinct("id").alias("portfolio_records"),
        F.countDistinct(
            F.when(F.lower(F.trim(F.col("gender"))) == "men", F.col("id"))
        ).alias("men_records"),
        F.countDistinct(
            F.when(F.lower(F.trim(F.col("gender"))) == "unisex", F.col("id"))
        ).alias("unisex_records"),
        F.countDistinct(
            F.when(F.lower(F.trim(F.col("gender"))) == "women", F.col("id"))
        ).alias("women_records"),
        F.countDistinct("dominant_accord").alias("distinct_dominant_accords"),
        F.countDistinct(
            F.when(
                F.col("dominant_accord").isNotNull()
                & F.col("second_accord").isNotNull(),
                F.struct("dominant_accord", "second_accord")
            )
        ).alias("distinct_ordered_dyads"),
        F.countDistinct(
            F.when(
                F.col("dominant_accord").isNotNull()
                & F.col("second_accord").isNotNull()
                & F.col("third_accord").isNotNull(),
                F.struct("dominant_accord", "second_accord", "third_accord")
            )
        ).alias("distinct_ordered_triads")
    )
    .orderBy("comparison_brand")
)

display(portfolio_overview)

total_portfolio_records = source_records.select("id").distinct().count()
print("Total records across the four portfolios:", total_portfolio_records)

In [ ]:
# 5. Accord frequency and concentration.

from pyspark.sql import Window

def build_profile_frequency(data, profile_columns, profile_level):
    valid = data.filter(
        F.expr(" AND ".join([f"{c} IS NOT NULL" for c in profile_columns]))
    )

    profile = F.concat_ws(" | ", *[F.col(c) for c in profile_columns])

    counts = (
        valid
        .withColumn("profile", profile)
        .groupBy("comparison_brand", "profile")
        .agg(F.countDistinct("id").alias("fragrance_count"))
    )

    brand_window = Window.partitionBy("comparison_brand")
    rank_window = Window.partitionBy("comparison_brand").orderBy(
        F.desc("fragrance_count"), F.asc("profile")
    )

    return (
        counts
        .withColumn(
            "portfolio_records_with_profile",
            F.sum("fragrance_count").over(brand_window)
        )
        .withColumn(
            "share_pct",
            F.round(
                F.col("fragrance_count")
                / F.col("portfolio_records_with_profile") * 100,
                2
            )
        )
        .withColumn("rank", F.row_number().over(rank_window))
        .withColumn(
            "cumulative_share_pct",
            F.round(
                F.sum("share_pct").over(
                    rank_window.rowsBetween(Window.unboundedPreceding, Window.currentRow)
                ),
                2
            )
        )
        .withColumn("profile_level", F.lit(profile_level))
        .select(
            "comparison_brand",
            "profile_level",
            "rank",
            "profile",
            "fragrance_count",
            "portfolio_records_with_profile",
            "share_pct",
            "cumulative_share_pct"
        )
        .orderBy("comparison_brand", "profile_level", "rank")
    )

dominant_frequency = build_profile_frequency(
    portfolio_profiles, ["dominant_accord"], "Dominant accord"
)

dyad_frequency = build_profile_frequency(
    portfolio_profiles,
    ["dominant_accord", "second_accord"],
    "Ordered dyad"
)

triad_frequency = build_profile_frequency(
    portfolio_profiles,
    ["dominant_accord", "second_accord", "third_accord"],
    "Ordered triad"
)

accord_frequency = (
    dominant_frequency
    .unionByName(dyad_frequency)
    .unionByName(triad_frequency)
)

display(accord_frequency)

# Top-five concentration by brand and accord level.
top_five_concentration = (
    accord_frequency
    .filter(F.col("rank") <= 5)
    .groupBy("comparison_brand", "profile_level")
    .agg(
        F.sum("fragrance_count").alias("top_five_records"),
        F.round(F.sum("share_pct"), 2).alias("top_five_share_pct")
    )
    .orderBy("comparison_brand", "profile_level")
)

display(top_five_concentration)

In [ ]:
# 6. Cross-brand overlap and exclusive accord profiles.

def build_profile_overlap(data, profile_columns, profile_level):
    valid = data.filter(
        F.expr(" AND ".join([f"{c} IS NOT NULL" for c in profile_columns]))
    )

    profile = F.concat_ws(" | ", *[F.col(c) for c in profile_columns])

    return (
        valid
        .withColumn("profile", profile)
        .groupBy("profile")
        .agg(
            F.countDistinct("comparison_brand").alias("brand_count"),
            F.sort_array(F.collect_set("comparison_brand")).alias("brands"),
            F.countDistinct("id").alias("source_record_count")
        )
        .withColumn("profile_level", F.lit(profile_level))
        .withColumn(
            "overlap_group",
            F.when(F.col("brand_count") == 1, "Exclusive to one brand")
            .when(F.col("brand_count") == 2, "Shared by two brands")
            .when(F.col("brand_count") == 3, "Shared by three brands")
            .otherwise("Shared by all four brands")
        )
        .select(
            "profile_level",
            "profile",
            "brand_count",
            "brands",
            "overlap_group",
            "source_record_count"
        )
    )

dominant_overlap = build_profile_overlap(
    portfolio_profiles, ["dominant_accord"], "Dominant accord"
)

dyad_overlap = build_profile_overlap(
    portfolio_profiles,
    ["dominant_accord", "second_accord"],
    "Ordered dyad"
)

triad_overlap = build_profile_overlap(
    portfolio_profiles,
    ["dominant_accord", "second_accord", "third_accord"],
    "Ordered triad"
)

profile_overlap = (
    dominant_overlap
    .unionByName(dyad_overlap)
    .unionByName(triad_overlap)
)

display(
    profile_overlap
    .orderBy("profile_level", F.desc("brand_count"), "profile")
)

# Number of profiles in each overlap group.
overlap_summary = (
    profile_overlap
    .groupBy("profile_level", "overlap_group")
    .agg(F.count("*").alias("profile_count"))
    .orderBy("profile_level", "overlap_group")
)

display(overlap_summary)

In [ ]:
# 7. Gender-specific portfolio size and accord diversity.

gender_diversity = (
    portfolio_profiles
    .filter(F.col("gender_key").isin("men", "unisex", "women"))
    .groupBy("comparison_brand", "gender_key")
    .agg(
        F.countDistinct("id").alias("portfolio_records"),
        F.countDistinct("dominant_accord").alias("distinct_dominant_accords"),
        F.countDistinct(
            F.when(
                F.col("dominant_accord").isNotNull()
                & F.col("second_accord").isNotNull(),
                F.struct("dominant_accord", "second_accord")
            )
        ).alias("distinct_ordered_dyads"),
        F.countDistinct(
            F.when(
                F.col("dominant_accord").isNotNull()
                & F.col("second_accord").isNotNull()
                & F.col("third_accord").isNotNull(),
                F.struct("dominant_accord", "second_accord", "third_accord")
            )
        ).alias("distinct_ordered_triads")
    )
    .orderBy("comparison_brand", "gender_key")
)

display(gender_diversity)

In [ ]:
# 8. Reconciliation checks against the four case-study portfolio counts.

expected_portfolio_counts = {
    "Hermès": 130,
    "Maison Francis Kurkdjian": 57,
    "Nasomatto": 14,
    "Nicolai Parfumeur Créateur": 75
}

expected_gender_counts = {
    "Hermès": {"men": 28, "unisex": 53, "women": 49},
    "Maison Francis Kurkdjian": {"men": 6, "unisex": 40, "women": 11},
    "Nasomatto": {"men": 2, "unisex": 9, "women": 3},
    "Nicolai Parfumeur Créateur": {"men": 8, "unisex": 38, "women": 29}
}

actual_portfolio_counts = {
    row["comparison_brand"]: row["portfolio_records"]
    for row in portfolio_overview.collect()
}

actual_gender_counts = {
    (row["comparison_brand"], row["gender_key"]): row["portfolio_records"]
    for row in gender_diversity.collect()
}

print("Portfolio count checks:")
for brand, expected in expected_portfolio_counts.items():
    actual = actual_portfolio_counts.get(brand, 0)
    print(f"{brand}: expected={expected}, actual={actual}, match={actual == expected}")

print("\nGender count checks:")
for brand, genders in expected_gender_counts.items():
    for gender, expected in genders.items():
        actual = actual_gender_counts.get((brand, gender), 0)
        print(
            f"{brand} / {gender}: expected={expected}, "
            f"actual={actual}, match={actual == expected}"
        )

print("\nExpected combined portfolio total:", sum(expected_portfolio_counts.values()))
print("Actual combined portfolio total:", total_portfolio_records)

# Overall gender totals for the four selected portfolios.
display(
    portfolio_profiles
    .filter(F.lower(F.trim(F.col("gender"))).isin("men", "unisex", "women"))
    .groupBy(F.lower(F.trim(F.col("gender"))).alias("gender"))
    .agg(F.countDistinct("id").alias("fragrance_count"))
    .orderBy("gender")
)